In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms                   
from torchvision.models import AlexNet_Weights, VGG16_Weights, ResNet50_Weights, EfficientNet_B0_Weights
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
import torchvision.models as models
import os
import time
import copy
from tqdm import tqdm

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/allikhankoshamet/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:45<00:00, 2.24MB/s]


In [6]:
data_dir = "/Users/allikhankoshamet/Desktop/dl_project/dl_new_dataset"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [3]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [4]:
import os
print("Current working dir:", os.getcwd())
print("Exists:", os.path.exists(data_dir))

Current working dir: /Users/allikhankoshamet/Desktop/dl_project/dl_new_dataset
Exists: True


In [5]:
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) 
                  for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=8, shuffle=True) 
               for x in ['train', 'val']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)

print(f"Class's find ✅: {num_classes}")
print(f"class's: {class_names}")
print(f"Train : {len(image_datasets['train'])}")
print(f"Val : {len(image_datasets['val'])}\n")

os.makedirs('models', exist_ok=True)

Class's find ✅: 15
class's: ['battery', 'copybook', 'espnder', 'headphones', 'highlighter', 'marker', 'mouse', 'napcin', 'notebook', 'paper holder', 'pen', 'pencile', 'stepler', 'sticker', 'usb']
Train : 214
Val : 39



In [8]:
def train_model(model, name, num_epochs=8):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    start = time.time()

    for epoch in range(num_epochs):
        print(f'\n=== Epoch {epoch+1}/{num_epochs} | {name} ===')
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in tqdm(dataloaders[phase], desc=phase):
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.float() / len(image_datasets[phase])

            print(f'{phase} Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        scheduler.step()

    time_elapsed = time.time() - start
    print(f'\n✅ {name} обучена за {time_elapsed//60:.0f}м {time_elapsed%60:.0f}с')
    print(f'Лучшая val accuracy: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    torch.save(model.state_dict(), f'models/{name}.pth')
    print(f'💾 Model saved for: models/{name}.pth\n')

In [ ]:
models_to_train = ['alexnet', 'vgg16', 'googlenet', 'resnet50', 'efficientnet_b0']

for name in models_to_train:
    print(f"\n🚀 Let's start {name.upper()} ...")

    if name == 'alexnet':
        model = models.alexnet(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == 'vgg16':
        model = models.vgg16(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == 'googlenet':
        model = models.googlenet(pretrained=True, aux_logits=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'resnet50':
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    train_model(model, name)

print("🎉!")

*Unfortently i already done with model alexnet, but suddenly file closed after my mistake, so initial code look like here, but below i don't call alexnet since i already since i saved it*

In [9]:
models_to_train = ['vgg16', 'googlenet', 'resnet50', 'efficientnet_b0']

for name in models_to_train:
    print(f"\n🚀 Let's start {name.upper()} ...")
    
    if name == 'vgg16':
        model = models.vgg16(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif name == 'googlenet':
        model = models.googlenet(pretrained=True, aux_logits=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'resnet50':
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'efficientnet_b0':
        model = models.efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    train_model(model, name)  
print("🎉!")


🚀 Let's start VGG16 ...


/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



=== Epoch 1/8 | vgg16 ===


train: 100%|██████████| 27/27 [00:37<00:00,  1.37s/it]


train Loss: 2.0579  Acc: 0.3879


...

train Loss: 0.2002  Acc: 0.9439


val: 100%|██████████| 5/5 [00:05<00:00,  1.10s/it]


val Loss: 0.0696  Acc: 1.0000

✅ vgg16 обучена за 5м 8с
Лучшая val accuracy: 1.0000
💾 Model saved for: models/vgg16.pth


🚀 Let's start GOOGLENET ...


/opt/homebrew/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ValueError: The parameter 'aux_logits' expected value True but got False instead.

In [10]:
models_to_train = ['googlenet', 'resnet50', 'efficientnet_b0']  

for name in models_to_train:
    print(f"\n🚀 Let's start {name.upper()} ...")
    
    if name == 'googlenet':
        model = models.googlenet(weights=None, aux_logits=False)        
        weights = models.GoogLeNet_Weights.IMAGENET1K_V1
        state_dict = weights.get_state_dict(progress=True)
        model.load_state_dict(state_dict, strict=False)         
        
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    train_model(model, name)  

print("🎉!")


🚀 Let's start GOOGLENET ...
Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /Users/allikhankoshamet/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


/opt/homebrew/lib/python3.10/site-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(
100%|██████████| 49.7M/49.7M [00:14<00:00, 3.59MB/s]



=== Epoch 1/8 | googlenet ===


train: 100%|██████████| 27/27 [00:38<00:00,  1.43s/it]


train Loss: 2.6065  Acc: 0.1449


...

val Loss: 0.8506  Acc: 0.8718

=== Epoch 8/8 | efficientnet_b0 ===


train: 100%|██████████| 27/27 [00:25<00:00,  1.05it/s]


train Loss: 1.0700  Acc: 0.7477


val: 100%|██████████| 5/5 [00:11<00:00,  2.28s/it]


val Loss: 0.7965  Acc: 0.8462

✅ efficientnet_b0 обучена за 4м 25с
Лучшая val accuracy: 0.8718
💾 Model saved for: models/efficientnet_b0.pth

🎉!
